# 날씨 기반 맞춤형 하루 계획 도우미

사용자의 지역·가능 시간·선호 활동을 바탕으로 현재 날씨를 조회하고 현실적인 하루 계획을 만듭니다.

실행 흐름: **Moderation → Responses API → Function Calling → Open-Meteo → Structured Output → 후속 대화/Streaming**

In [2]:
# 기본 세팅
import json
import os
import urllib.parse
import urllib.request
from pathlib import Path

from IPython.display import Audio, display
from openai import OpenAI
from pydantic import BaseModel, Field

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY 환경 변수를 먼저 설정하세요.")

client = OpenAI(max_retries=2, timeout=30.0)
API_MODEL = "gpt-5.6-luna"
print("OpenAI 클라이언트 준비 완료")

OpenAI 클라이언트 준비 완료


## 강의 내용과 프로젝트 연결

| 배운 기능 | 프로젝트 적용 |
|---|---|
| Responses API | 계획 생성 |
| `previous_response_id` | 후속 요청에서 이전 계획 기억 |
| Function calling | Open-Meteo 현재 날씨 조회 |
| Structured Output | 일정·장소·준비물을 Pydantic 객체로 반환 |
| Streaming | 완성된 계획을 자연어로 실시간 설명 |
| `instructions` | 안전하고 현실적인 계획 도우미 역할 설정 |
| Usage / Prompt cache | 누적 토큰과 캐시 토큰 확인 |
| Moderation | 사용자 입력 안전 검사 |
| TTS | 계획을 음성 브리핑 파일로 저장 |

## 1. 구조화된 출력 정의

In [3]:
class Activity(BaseModel):
    time: str = Field(description="활동 시작 시각 또는 시간대")
    title: str = Field(description="활동 이름")
    location: str = Field(description="추천 장소 또는 실내/실외 구분")
    reason: str = Field(description="날씨와 사용자 조건을 고려한 추천 이유")
    preparation: list[str] = Field(description="필요한 준비물")


class DailyPlan(BaseModel):
    city: str
    weather_summary: str
    activities: list[Activity]
    caution: str = Field(description="날씨 또는 안전 관련 주의사항")

## 2. 날씨 함수와 Function Tool

Open-Meteo는 API 키 없이 사용할 수 있습니다. 모델은 도시의 좌표를 함수 인자로 결정하고, 실제 날씨 조회는 아래 파이썬 함수가 수행합니다.

In [4]:
def get_weather(city: str, latitude: float, longitude: float) -> str:
    params = urllib.parse.urlencode({
        "latitude": latitude,
        "longitude": longitude,
        "current": ("temperature_2m,apparent_temperature,relative_humidity_2m,"
                    "precipitation,weather_code,wind_speed_10m,is_day"),
        "timezone": "auto",
    })
    url = f"https://api.open-meteo.com/v1/forecast?{params}"
    with urllib.request.urlopen(url, timeout=10) as response:
        data = json.load(response)
    result = {
        "city": city,
        "timezone": data.get("timezone"),
        **data["current"],
        "units": data.get("current_units", {}),
    }
    return json.dumps(result, ensure_ascii=False)


WEATHER_TOOLS = [{
    "type": "function",
    "name": "get_weather",
    "description": "도시의 현재 기온, 체감온도, 습도, 강수량, 날씨 코드, 풍속을 조회한다.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "도시 이름"},
            "latitude": {"type": "number", "description": "도시 중심의 위도"},
            "longitude": {"type": "number", "description": "도시 중심의 경도"}
        },
        "required": ["city", "latitude", "longitude"],
        "additionalProperties": False
    },
    "strict": True
}]

## 3. 대화형 계획 도우미

`ask()`는 입력을 검사하고, 날씨 함수 호출 결과를 모델에 돌려준 뒤 `DailyPlan`으로 파싱합니다. 마지막 응답 ID를 저장하므로 다음 요청에서 앞 계획을 기억합니다.

In [5]:
PLANNER_INSTRUCTIONS = """
당신은 날씨 기반 하루 계획 도우미입니다.
사용자의 지역, 가능한 시간, 선호 활동을 반영해 현실적인 계획을 만드세요.
반드시 도구가 반환한 현재 날씨만 날씨 판단의 근거로 사용하세요.
강수, 폭염, 한파, 강풍 가능성이 있으면 무리한 야외 활동을 피하고 대안을 제안하세요.
사용자가 밝히지 않은 취향이나 건강 상태를 사실처럼 단정하지 마세요.
응답은 요청된 DailyPlan 스키마를 정확히 따르세요.
""".strip()


class WeatherPlanner:
    def __init__(self):
        self.previous_response_id = None
        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.total_cached_tokens = 0
        self.last_plan = None

    def _moderate(self, text: str) -> None:
        result = client.moderations.create(
            model="omni-moderation-latest", input=text
        ).results[0]
        if result.flagged:
            raise ValueError("안전 검사를 통과하지 못한 요청입니다. 표현을 바꿔 다시 시도해 주세요.")

    def _record_usage(self, response) -> None:
        if response.usage is None:
            return
        self.total_input_tokens += response.usage.input_tokens
        self.total_output_tokens += response.usage.output_tokens
        details = response.usage.input_tokens_details
        self.total_cached_tokens += getattr(details, "cached_tokens", 0) or 0

    def ask(self, user_request: str) -> DailyPlan:
        self._moderate(user_request)
        tool_response = client.responses.create(
            model=API_MODEL,
            instructions=PLANNER_INSTRUCTIONS,
            input=user_request,
            previous_response_id=self.previous_response_id,
            tools=WEATHER_TOOLS,
            tool_choice={"type": "function", "name": "get_weather"},
            parallel_tool_calls=False,
            prompt_cache_key="weather-planner-v1",
            safety_identifier="mission-notebook-user",
            metadata={"project": "weather-daily-planner", "step": "weather"}
        )
        self._record_usage(tool_response)

        tool_outputs = []
        for item in tool_response.output:
            if item.type != "function_call":
                continue
            if item.name != "get_weather":
                raise ValueError(f"지원하지 않는 도구입니다: {item.name}")
            arguments = json.loads(item.arguments)
            weather_json = get_weather(**arguments)
            tool_outputs.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": weather_json
            })

        if not tool_outputs:
            raise RuntimeError("모델이 날씨 함수를 호출하지 않았습니다.")

        final_response = client.responses.parse(
            model=API_MODEL,
            instructions=PLANNER_INSTRUCTIONS,
            previous_response_id=tool_response.id,
            input=tool_outputs,
            tools=WEATHER_TOOLS,
            tool_choice="none",
            text_format=DailyPlan,
            prompt_cache_key="weather-planner-v1",
            safety_identifier="mission-notebook-user",
            metadata={"project": "weather-daily-planner", "step": "plan"}
        )
        self._record_usage(final_response)
        self.previous_response_id = final_response.id
        self.last_plan = final_response.output_parsed
        return self.last_plan

    def stream_briefing(self, request: str = "방금 계획을 친근한 한국어 브리핑으로 설명해줘."):
        if self.previous_response_id is None:
            raise RuntimeError("먼저 ask()로 계획을 만들어 주세요.")
        self._moderate(request)
        with client.responses.stream(
            model=API_MODEL,
            instructions="이전 계획을 바탕으로 간결하고 친근한 한국어 브리핑을 작성하세요.",
            input=request,
            previous_response_id=self.previous_response_id,
            prompt_cache_key="weather-planner-v1",
            safety_identifier="mission-notebook-user"
        ) as stream:
            for event in stream:
                if event.type == "response.output_text.delta":
                    print(event.delta, end="", flush=True)
            response = stream.get_final_response()
        print()
        self._record_usage(response)
        self.previous_response_id = response.id
        return response.output_text

    def usage(self) -> dict:
        return {
            "input_tokens": self.total_input_tokens,
            "output_tokens": self.total_output_tokens,
            "cached_tokens": self.total_cached_tokens,
            "total_tokens": self.total_input_tokens + self.total_output_tokens
        }

    def reset(self) -> None:
        self.previous_response_id = None
        self.last_plan = None

## 4. 실행해 보기

아래 요청을 원하는 도시·시간·활동으로 바꿔 보세요. 한 번 실행할 때 OpenAI API와 Open-Meteo 네트워크 연결이 필요합니다.

In [10]:
planner = WeatherPlanner()
plan = planner.ask("부산에 살고 오늘 저녁 9시부터 2시간 동안 가볍게 운동하고 싶어.")
display(plan)

DailyPlan(city='부산', weather_summary='현재 맑고 강수량은 0.0mm입니다. 기온은 29.7°C, 체감온도는 34.8°C, 습도는 74%, 풍속은 8.7km/h입니다.', activities=[Activity(time='21:00~21:10', title='가벼운 준비운동', location='실내', reason='현재 체감온도와 습도가 높아 야외 운동보다 실내에서 천천히 몸을 데우는 편이 안전합니다.', preparation=['물', '편한 운동복', '미끄럼 방지 운동화 또는 요가 매트']), Activity(time='21:10~21:45', title='저강도 유산소 운동', location='실내', reason='실내 자전거, 가벼운 스텝 운동, 저강도 홈트레이닝처럼 강도를 조절하기 쉬운 운동을 권합니다.', preparation=['물', '수건', '환기 또는 냉방이 가능한 공간']), Activity(time='21:45~22:00', title='가벼운 근력운동', location='실내', reason='스쿼트, 벽 짚고 팔굽혀펴기, 브리지 등 무리하지 않는 맨몸운동으로 구성할 수 있습니다.', preparation=['요가 매트', '필요하면 낮은 강도의 밴드']), Activity(time='22:00~22:20', title='저강도 유산소 또는 스트레칭', location='실내', reason='남은 시간은 강도를 낮춰 관절 가동성 운동과 전신 스트레칭 중심으로 마무리하세요.', preparation=['물', '수건']), Activity(time='22:20~23:00', title='정리운동 및 휴식', location='실내', reason='높은 체감온도와 습도를 고려해 충분히 호흡과 체온을 안정시키는 시간으로 마무리하는 것이 좋습니다.', preparation=['물 또는 전해질 음료', '마른 옷'])], caution='현재 체감온도가 34.8°C이고 습도가 74%이므로 야외 달리기나 

In [11]:
# previous_response_id로 앞 계획을 기억하면서 다시 날씨를 확인해 수정
revised_plan = planner.ask("야외 활동은 빼고, 같은 시간 안에서 실내 계획으로 바꿔줘.")
display(revised_plan)

DailyPlan(city='부산', weather_summary='현재 맑고 강수량은 0.0mm입니다. 기온은 29.7°C, 체감온도는 34.8°C, 습도는 74%, 풍속은 8.7km/h입니다.', activities=[Activity(time='21:00~21:10', title='준비운동', location='실내', reason='현재 체감온도와 습도가 높으므로 냉방이나 환기가 가능한 실내에서 천천히 몸을 데우는 것이 좋습니다.', preparation=['물', '편한 운동복', '요가 매트 또는 미끄럼 방지 매트']), Activity(time='21:10~21:40', title='저강도 유산소 운동', location='실내', reason='실내 자전거, 가벼운 제자리 걷기, 저강도 홈트레이닝처럼 강도를 쉽게 조절할 수 있는 운동을 권합니다.', preparation=['물', '수건', '냉방 또는 환기가 가능한 공간']), Activity(time='21:40~22:10', title='가벼운 맨몸 근력운동', location='실내', reason='스쿼트, 브리지, 벽 짚고 팔굽혀펴기 등을 무리하지 않는 강도로 진행하세요.', preparation=['요가 매트', '필요하면 낮은 강도의 운동 밴드', '물']), Activity(time='22:10~22:30', title='스트레칭 및 유연성 운동', location='실내', reason='하체와 어깨, 허리를 중심으로 천천히 스트레칭해 운동 강도를 낮추고 몸을 이완하세요.', preparation=['요가 매트', '수건']), Activity(time='22:30~23:00', title='정리운동 및 휴식', location='실내', reason='높은 체감온도와 습도를 고려해 호흡과 체온을 충분히 안정시키며 마무리하는 시간입니다.', preparation=['물 또는 전해질 음료', '마른 옷', '수건'])], caution='전체 일정을 실내 활동으로 구성했습니다. 현

In [14]:
# 자연어 설명을 스트리밍으로 출력
briefing_text = planner.stream_briefing()

# 지금까지 사용한 토큰 확인
planner.usage()

오늘 밤 **9시부터 11시까지는 시원한 실내에서 가볍게 운동**해보세요.

먼저 10분 동안 제자리 걷기와 관절 돌리기로 몸을 풀고, 30분간 실내 자전거나 저강도 홈트레이닝을 진행합니다. 이어서 30분은 스쿼트·브리지·벽 짚고 팔굽혀펴기 같은 맨몸운동을 해주세요. 그다음 20분은 전신 스트레칭으로 몸을 이완하고, 마지막 30분은 천천히 호흡을 가다듬으며 정리운동과 휴식으로 마무리하면 됩니다.

운동 중에는 물을 자주 마시고, 실내 온도와 환기를 적절히 조절하세요. 몸이 무겁거나 어지러우면 무리하지 말고 바로 쉬어주세요.


{'input_tokens': 10618,
 'output_tokens': 2052,
 'cached_tokens': 4016,
 'total_tokens': 12670}

## 5. 선택 기능: 음성 브리핑

강의에서 사용한 TTS 모델로 스트리밍 결과를 MP3 파일에 저장합니다.

In [15]:
def save_voice_briefing(text: str, filename: str = "daily_plan_busan.mp3") -> Path:
    speech = client.audio.speech.create(
        model="gpt-4o-mini-tts",
        voice="echo",
        instructions="차분하고 또렷한 한국어 안내 음성으로 읽어 주세요.",
        input=text
    )
    path = Path(filename)
    path.write_bytes(speech.content)
    return path


audio_path = save_voice_briefing(briefing_text)
display(Audio(filename=str(audio_path)))

## 실험 아이디어

- 같은 요청을 여러 번 실행하고 `cached_tokens`가 어떻게 달라지는지 비교하기
- `text.verbosity` 또는 추론 강도를 바꿔 토큰과 결과 품질 비교하기
- Open-Meteo 연결 실패나 API 오류에 대한 `try/except`와 재시도 UI 추가하기
- `planner.reset()`으로 새 대화를 시작한 뒤 기억 여부 비교하기

참고: [GPT-5.6 Luna](https://developers.openai.com/api/docs/models/gpt-5.6-luna), [Function calling](https://developers.openai.com/api/docs/guides/function-calling)